In [1]:
import json
import os
import pickle

import numpy as np
# import pandas as pd
import torch
from B0_MODEL import UNET
from B1_UTILS import (identify_chain, load_checkpoint, map_normalizing,
                      map_writing, predict_location, smooth_location)
from B2_UTILS_modeling import refine_mass_centers, write_pdb  # track_5_3,

In [2]:
# ====================== Hyperparameters etc. ========================
MAIN_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST'
MODEL_TYPES = ['secondary_strctures', 'key_atoms', 'residue_types']
MAP_DIRS_TYPES = [['map_sample'],
                  ['map_sample', 'secondary_strctures_pred_prob'],
                  [
                      'map_sample', 'secondary_strctures_pred_prob',
                      'key_atoms_pred_prob'
                  ]]
# IN_CHANNELS = [1, 6, 10]

data_path = os.path.join(MAIN_PATH, 'train_val_data')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# batch_size (vs GPU Memory: 1vs2877; 2vs3485; 4vs4668; 8vs7162)
batch_size = 8

num_workers = 2
learning_rate = 1e-4
pin_memory = True

In [3]:
def initial_setting(model_index, data_path, device):
    """
    This function performs the initial setup before training process:
    1. Read the class weights of the UNET model,
    2. Read the saved epoch number,
    3. Generate paths of file that will be loaded to UNET model.
    """

    model_type = MODEL_TYPES[model_index]
    # Load the class weights of all model types
    weight_path = os.path.join(data_path, 'class_weight_for_training.txt')
    try:
        # Load the weight for training
        with open(weight_path, "r") as file:
            weight_all = json.load(file)

        if model_type not in weight_all.keys():
            print(f"Weight for {model_type} is not found.")
            exit()
        else:
            c_w = weight_all[model_type]
            class_weights = torch.FloatTensor(c_w).to(device)

            # Calculate the number of output channels based on the number of tags in model_type
            out_channels = len(c_w)
            in_channels = 1
            if model_index > 0:
                for ii in range(model_index):
                    in_channels = in_channels + len(
                        weight_all[MODEL_TYPES[ii]])

    except FileNotFoundError:
        print("Weight file does not exist.")
        exit()

    # Load the load_epoch_num data or initialize it if the file doesn't exist
    load_epoch_num_path = os.path.join(data_path, "epoch_num_path.txt")

    if not os.path.exists(load_epoch_num_path):
        load_epoch_num_all = {model_type: 0}
    else:
        with open(load_epoch_num_path, "r") as file:
            load_epoch_num_all = json.load(file)
            if model_type not in load_epoch_num_all.keys():
                load_epoch_num_all[model_type] = 0
    load_epoch_num = load_epoch_num_all[model_type]

    # Save the updated load_epoch_num_all data
    with open(load_epoch_num_path, "w") as file:
        json.dump(load_epoch_num_all, file)

    load_model_path = os.path.join(
        data_path, 'checkpoint', model_type,
        f'{model_type}.epoch{load_epoch_num}.pth.tar')

    return class_weights, in_channels, out_channels, load_epoch_num, load_model_path


In [5]:
def output(out_channels, part_possibility, model_data, model_type,
           zoom_factors):

    for channel_num in range(1, out_channels):
        print(f"\n==> Start map writing of {model_type}: ")

        possibility = part_possibility[channel_num, :, :, :]
        out_map_file = f"{map_name.split('.mrc')[0]}_{model_type}_{channel_num}_possibility.mrc"
        out_map = os.path.join(folder_path, "possibility", out_map_file)
        map_writing(MAP_PATH, possibility, out_map, zoom_factors)

        label_data = np.where(model_data == channel_num, 1, 0)
        out_map_file = f"{map_name.split('.mrc')[0]}_{model_type}_{channel_num}.mrc"
        out_map = os.path.join(folder_path, "unrefined", out_map_file)
        map_writing(MAP_PATH, label_data, out_map, zoom_factors, is_int=True)

        # smooth_loc = smooth_location(label_data, channel_num)
        # out_map_file = f"{map_name.split('.mrc')[0]}_{model_type}_{channel_num}_refined.mrc"
        # out_map = os.path.join(folder_path, "refined", out_map_file)
        # map_writing(MAP_PATH, smooth_loc, out_map, zoom_factors, is_int=True)

In [4]:
RAW_DATA_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw'
MAP_PATH = os.path.join(RAW_DATA_PATH, 'EMD-0065_re_3.27', 'emd_0065.map')
folder_path = os.path.dirname(MAP_PATH)
map_name = os.path.basename(MAP_PATH)
os.makedirs(os.path.join(folder_path, "possibility"), exist_ok=True)
os.makedirs(os.path.join(folder_path, "unrefined"), exist_ok=True)
os.makedirs(os.path.join(folder_path, "refined"), exist_ok=True)
os.makedirs(os.path.join(folder_path, "chains"), exist_ok=True)
os.makedirs(os.path.join(folder_path, "temp_in_chain"), exist_ok=True)
map_data, zoom_factors = map_normalizing(MAP_PATH)


In [5]:
model_index = 0
model_type = MODEL_TYPES[model_index]
class_weights, in_channels, out_channels, load_epoch_num, load_model_path = initial_setting(
    model_index, data_path, device=DEVICE)
print(f"\n==> Start {model_type} prediction: ")
model = UNET(in_channels, out_channels).to(DEVICE)
load_checkpoint(load_model_path, model, map_location=DEVICE)



==> Start secondary_strctures prediction: 


In [ ]:
# part_possibility = predict_location(map_data, model, out_channels, DEVICE)
# model_data = np.argmax(part_possibility, 0)

In [19]:
# Iterate over 3D cubes of the map data
# Seprate the map into 64*64*64 cubes but overlaps with 7 for every dimemsion

map_shape = map_data.shape  #(?, 200, 200, 200)
if len(map_data.shape) == 3:
    new_map_data = np.expand_dims(map_data, axis=0)
else:
    new_map_data = map_data




######### stop here 3/7/2024
######### the idea: new_num = new_shape//50 how to accomplish it?
new_shape = new_map_data.shape
new_num = [new_shape[i] // 50 for i in range(len(new_shape))]
new_num

[0, 4, 4, 4]

In [ ]:

cube_size = 50
num_cubes = (new_shape[-3] // cube_size) * (new_shape[-2] // cube_size) * (new_shape[-1] // cube_size)

# Add 7 of zeros to every begining of the last 3 dimemsions
pad_width = ((0, 0), (7, 0), (7, 0), (7, 0))
new_map_data = np.pad(new_map_data,
                      pad_width,
                      mode='constant',
                      constant_values=0)

new_map_data = torch.from_numpy(new_map_data)
model.eval()
# Iterate over 3D cubes of the map data
with torch.no_grad():
    num_completed = 0


In [ ]:
def predict_location(map_data, model, out_channels, device):

    new_shape_map, insert_idx = map_reshape(map_data)
    new_shape = new_shape_map.shape
    new_shape_map = torch.from_numpy(new_shape_map)

    model.eval()
    # Iterate over 3D cubes of the map data
    with torch.no_grad():
        cube_size = 50
        num_cubes = (new_shape[-3] // cube_size) * (
            new_shape[-2] // cube_size) * (new_shape[-1] // cube_size)
        num_completed = 0
        for zz in range(new_shape[-3] // cube_size):
            for yy in range(new_shape[-2] // cube_size):
                for xx in range(new_shape[-1] // cube_size):
                    # Extract a 3D cube from the map data
                    if len(new_shape) == 3:
                        cube = new_shape_map[None, None, cube_size * zz:64 +
                                             cube_size * zz, cube_size *
                                             yy:64 + cube_size * yy,
                                             cube_size * xx:64 +
                                             cube_size * xx]
                    else:
                        cube = new_shape_map[None, :, cube_size * zz:64 +
                                             cube_size * zz, cube_size *
                                             yy:64 + cube_size * yy,
                                             cube_size * xx:64 +
                                             cube_size * xx]

                    # Move the cube to the specified device and perform prediction
                    cube = cube.to(device)
                    # a = model(cube)
                    # print(f'model shape is {a.shape}')

                    pred_data = torch.softmax(model(cube), 1)

                    # Write predicted data to new_shape_model
                    model_shape = new_shape[-3:]
                    model_shape = np.insert(model_shape, 0, out_channels)
                    new_shape_model = torch.zeros(tuple(model_shape))
                    new_shape_model[:, 7 + cube_size * zz:57 + cube_size * zz,
                                    7 + cube_size * yy:57 + cube_size * yy,
                                    7 + cube_size * xx:57 +
                                    cube_size * xx] = pred_data[0, :, 7:57,
                                                                7:57, 7:57]

                    # Update progress
                    num_completed += 1
                    print(
                        f"{num_completed}/{num_cubes} of prediction is done.",
                        end='\r')
        # print("\n")

    # Extract model data
    part_possibility = new_shape_model[:, insert_idx[0, 0]:insert_idx[0, 1],
                                       insert_idx[1, 0]:insert_idx[1, 1],
                                       insert_idx[2, 0]:insert_idx[2,
                                                                   1]].numpy()

    return part_possibility

In [ ]:
model_index = 1
model_type = MODEL_TYPES[model_index]
class_weights, in_channels, out_channels, load_epoch_num, load_model_path = initial_setting(
    model_index, data_path, device=DEVICE)
print(f"\n==> Start {model_type} prediction: ")
model = UNET(in_channels, out_channels).to(DEVICE)
load_checkpoint(load_model_path, model, map_location=DEVICE)
map_data = np.expand_dims(map_data, axis=0)
map_data = np.concatenate((map_data, part_possibility), axis=0)
part_possibility = predict_location(map_data, model, out_channels, DEVICE)
model_data = np.argmax(part_possibility, 0)

In [ ]:
output(out_channels, part_possibility, model_data, model_type, zoom_factors)

In [ ]:
model_index = 2
model_type = MODEL_TYPES[model_index]
class_weights, in_channels, out_channels, load_epoch_num, load_model_path = initial_setting(
    model_index, data_path, device=DEVICE)
print(f"\n==> Start {model_type} prediction: ")
model = UNET(in_channels, out_channels).to(DEVICE)
load_checkpoint(load_model_path, model, map_location=DEVICE)
# map_data = np.expand_dims(map_data, axis=0)
map_data = np.concatenate((map_data, part_possibility), axis=0)
part_possibility = predict_location(map_data, model, out_channels, DEVICE)
model_data = np.argmax(part_possibility, 0)

In [ ]:
output(out_channels, part_possibility, model_data, model_type, zoom_factors)